# ICS 3202 - Practical Exercise 3
## Uniform Cost Search: self-driving trailer, Nairobi -> Kisumu

Uses the directed graph and path costs from Part 1 (design.md, Section 4.1).
Edge costs are road distances in km. The search expands the cheapest frontier node first, and the goal test runs on expansion.


## Run the search
The output below is from running the program. Running this cell reproduces it.

In [1]:
"""
ICS 3202 - Artificial Intelligence, Practical Exercise 3
Uniform Cost Search (UCS) for the self-driving trailer: Nairobi -> Kisumu.

The graph is the directed graph from Part 1 (design.md, Section 4.1).
Each edge cost is the road distance in km. Only the directions drawn in the
graph are used (Nairobi towards Kisumu).

Run with:  python ucs_trailer.py
"""

import heapq
import itertools
import math

# Directed graph: town -> {next town: distance in km}
GRAPH = {
    "Nairobi":  {"Naivasha": 88, "Narok": 142},
    "Naivasha": {"Nakuru": 71},
    "Nakuru":   {"Eldoret": 156, "Molo": 47},
    "Eldoret":  {"Kapsabet": 48},
    "Kapsabet": {"Kericho": 74},
    "Molo":     {"Kericho": 60},
    "Narok":    {"Bomet": 86},
    "Bomet":    {"Kisii": 68},
    "Kisii":    {"Kisumu": 83},
    "Kericho":  {"Kisumu": 83},
    "Kisumu":   {},
}

START = "Nairobi"
GOAL = "Kisumu"
EXPECTED_COST = 349  # Route 1 in the report


def uniform_cost_search(graph, start, goal, verbose=True):
    """
    Uniform Cost Search (graph version).

    The frontier is a priority queue ordered by path cost g(n), the total
    distance from the start. The goal test is applied when a node is taken
    off the frontier (expanded), not when it is generated. This is what makes
    the returned path optimal.

    Returns (path, cost). If the goal is unreachable, returns (None, None).
    """
    counter = itertools.count()  # tie-breaker so heapq never compares lists
    frontier = [(0, next(counter), start, [start])]  # (g, tie, node, path)
    best_cost = {start: 0}       # cheapest cost found so far for each node
    explored = set()
    step = 0

    if verbose:
        print(f"Start: {start}    Goal: {goal}")
        print("-" * 78)

    while frontier:
        g, _, node, path = heapq.heappop(frontier)

        # Skip stale entries: a cheaper route to this node was already expanded.
        if node in explored:
            continue

        explored.add(node)
        step += 1

        if verbose:
            live = sorted(
                (c, n) for c, _, n, _ in frontier
                if n not in explored and best_cost.get(n) == c
            )
            frontier_text = ", ".join(f"{n} ({c})" for c, n in live) or "(empty)"
            print(f"Step {step:2}: expand {node:<9} g = {g:>3} km   "
                  f"path: {' -> '.join(path)}")
            print(f"          frontier (before expanding): {frontier_text}")

        # Goal test on expansion
        if node == goal:
            return path, g

        # Expand: generate children and push them with their path costs
        for child, km in graph[node].items():
            if child in explored:
                continue
            new_cost = g + km
            if new_cost < best_cost.get(child, math.inf):
                best_cost[child] = new_cost
                heapq.heappush(frontier,
                               (new_cost, next(counter), child, path + [child]))

    return None, None


def print_cost_breakdown(graph, path):
    """Show each segment of the path and the running total."""
    print("\nCost of the optimal path:")
    total = 0
    for a, b in zip(path, path[1:]):
        km = graph[a][b]
        total += km
        print(f"  d({a}, {b}) = {km} km")
    print(f"  Total = {' + '.join(str(graph[a][b]) for a, b in zip(path, path[1:]))}"
          f" = {total} km")
    return total


def main():
    print("UNIFORM COST SEARCH: self-driving trailer, Nairobi -> Kisumu")
    print("=" * 78)

    path, cost = uniform_cost_search(GRAPH, START, GOAL, verbose=True)

    print("=" * 78)
    if path is None:
        print("No route from", START, "to", GOAL)
        return

    print(f"Optimal path : {' -> '.join(path)}")
    total = print_cost_breakdown(GRAPH, path)
    print(f"Path cost    : {cost} km")

    if total == cost and cost == EXPECTED_COST:
        print(f"Check        : matches Route 1 in the report ({EXPECTED_COST} km)")
    else:
        print(f"Check        : does NOT match the report ({EXPECTED_COST} km)")


if __name__ == "__main__":
    main()


UNIFORM COST SEARCH: self-driving trailer, Nairobi -> Kisumu
Start: Nairobi    Goal: Kisumu
------------------------------------------------------------------------------
Step  1: expand Nairobi   g =   0 km   path: Nairobi
          frontier (before expanding): (empty)
Step  2: expand Naivasha  g =  88 km   path: Nairobi -> Naivasha
          frontier (before expanding): Narok (142)
Step  3: expand Narok     g = 142 km   path: Nairobi -> Narok
          frontier (before expanding): Nakuru (159)
Step  4: expand Nakuru    g = 159 km   path: Nairobi -> Naivasha -> Nakuru
          frontier (before expanding): Bomet (228)
Step  5: expand Molo      g = 206 km   path: Nairobi -> Naivasha -> Nakuru -> Molo
          frontier (before expanding): Bomet (228), Eldoret (315)
Step  6: expand Bomet     g = 228 km   path: Nairobi -> Narok -> Bomet
          frontier (before expanding): Kericho (266), Eldoret (315)
Step  7: expand Kericho   g = 266 km   path: Nairobi -> Naivasha -> Nakuru -> Molo ->